### Structured Output

### Pydantic

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
model = init_chat_model("groq:openai/gpt-oss-120b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016057991A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016057992510>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="In This year the movie released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The rating of the movie is")


In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016057991A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016057992510>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'pa

In [6]:
model_with_structure.invoke("Provide me detail of movie Rush hour")

Movie(title='Rush Hour', year=1998, director='Brett Ratner', rating=7.0)

### Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
   """A movie with Details"""
   title:str = Field(..., description="The Title of the movie")
   year:str = Field(...,description="The year the movie was released")
   director:str = Field(...,description="the director of the movie")
   rating:str = Field(...,description="the movies rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke(
    "Provide me details about the movie Inception"
)
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie Inception. Use the function Movie with director, rating, title, year. Provide info. Probably rating is out of 10, maybe 8.8. Director Christopher Nolan, year 2010. Title "Inception". Use function.', 'tool_calls': [{'id': 'fc_a35ee97a-85bf-476e-8a97-98fc626502af', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":"2010"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 164, 'total_tokens': 275, 'completion_time': 0.230591779, 'completion_tokens_details': {'reasoning_tokens': 59}, 'prompt_time': 0.008081477, 'prompt_tokens_details': None, 'queue_time': 0.369923856, 'total_time': 0.238673256}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_prov

### Nested Structure

In [10]:
from pydantic import BaseModel,Field 
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    gener:list[str]
    budget:float | None = Field(None, description="Budget in million usd")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide Details about the movie Inception")
response 

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], gener=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A Movie Details"""
    title: Annotated[str,...,"The Title of the movie"]
    year: Annotated[str,...,"The year The movie is released"]
    director: Annotated[str,...,"The Director of the movie"]
    rating: Annotated[float,...,"The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Please provide the details of the movie avangers")
response

{'director': 'Joss Whedon',
 'rating': 8,
 'title': 'The Avengers',
 'year': '2012'}